# ✈ Airline Operational Risk EDA

## Problem Statement

Airlines operate in a cost-sensitive environment where even minor schedule disruptions can cascade into large financial and reputational impacts. The goal is to minimize operational uncertainty by leveraging data analytics to detect patterns of instability and
reduce disruption-related losses

## Data Definition

- **YEAR**  
  Year of the flight operation.

- **QUARTER**  
  Quarter of the year (1–4).

- **MONTH**  
  Month of operation (1–12).

- **DAY_OF_WEEK**  
  Day of the week (1 = Monday, 7 = Sunday).


- **MKT_UNIQUE_CARRIER**  
  Unique airline carrier code (e.g., AA, DL, UA).

- **ORIGIN_AIRPORT_ID**  
  Unique numeric identifier of departure airport.

- **ORIGIN**  
  Departure airport code (e.g., JFK, LAX).

- **DEST_AIRPORT_ID**  
  Unique numeric identifier of destination airport.

- **DEST**  
  Destination airport code.


- **CRS_DEP_TIME**  
  Scheduled departure time (in HHMM format).

- **DEP_TIME_BLK**  
  Scheduled departure time block (e.g., 0800–0859).

- **TAXI_OUT**  
  Time (in minutes) aircraft spent taxiing before takeoff.

- **TAXI_IN**  
  Time (in minutes) aircraft spent taxiing after landing.


- **CANCELLED**  
  Binary indicator (1 = Flight cancelled, 0 = Not cancelled).

- **DIVERTED**  
  Binary indicator (1 = Flight diverted, 0 = Not diverted).


- **ACTUAL_ELAPSED_TIME**  
  Total actual flight duration (gate-to-gate) in minutes.

- **AIR_TIME**  
  Actual time spent in air (in minutes).

- **DISTANCE**  
  Distance between origin and destination airports (in miles).

- **DISTANCE_GROUP**  
  Categorized distance interval group.


- **CARRIER_DELAY**  
  Delay caused by airline operations.

- **WEATHER_DELAY**  
  Delay due to weather conditions.

- **NAS_DELAY**  
  Delay due to National Airspace System (traffic congestion).

- **SECURITY_DELAY**  
  Delay due to security-related issues.

- **LATE_AIRCRAFT_DELAY**  
  Delay caused by late arrival of the same aircraft.


- **OPERATIONAL_RISK**  
  Risk classification label indicating operational risk level of the flight.

## Table of Contents


1. **[Import Libraries](#import_lib)** 
2. **[Set Options](#set_options)** 
3. **[Read Data](#Read_Data)** 
4. **[Understand  and Prepare the Data](#Understand_Data)**
5. **[Understand the variables](#Understanding_variables)**
6. **[Check for Missing Values](#missing)**
7. **[Study Correlation](#correlation)**
8. **[Detect Outliers](#outliers)**
9. **[Create a new variable 'region'](#region)**
10. **[Some more analysis](#more)** 


# Importing Libraries


In [69]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score,recall_score,r2_score,precision_score,roc_auc_score,confusion_matrix

import warnings 
warnings.filterwarnings("ignore")

In [ ]:
PROJECT_ROOT = Path.cwd().parent
data_path = PROJECT_ROOT / "data" / "train_sample_2024.csv"
data=pd.read_csv(data_path)
df=data.copy()

# Data Understanding

In [ ]:
# Checking the number of rows and columns
df.shape

In [ ]:
# Checking out the basic info of the data
df.info()

In [ ]:
# Checking duplicates
df.duplicated().sum()

In [ ]:
# displaying top 5 data
df.head()

### INFERENCE
**1) The dataset has 2 Lakh rows and 25 columns.**

**2) There 16 numerical columns and 9 categorical columns.**

**3) There are no duplicate rows present in the dataset.**


## Understanding the variables

In [ ]:
for i in df.columns:
    print(i)
    print(df[i].unique())
    print()

In [ ]:
# Separating numerical and categorical columns
num=df.select_dtypes(include=np.number).columns.to_list()
cat=df.select_dtypes(include=object).columns.to_list()

In [ ]:
# Summary statistic of numerical variables
df[num].describe().T

### INFERENCE
**1) OPERATIONAL_RISK**: About 20.6% of flights are risky, so the data is imbalanced with most flights being normal.

**2) ACTUAL_ELAPSED_TIME**: Average flight time is 138 minutes with high variation, suggesting longer flights may have higher operational complexity.

**3) CARRIER_DELAY**: Most flights have no delay, but a few have very large delays, indicating rare but high-impact events.


In [ ]:
# Summary statistic of categorical variables
df[cat].describe().T


### INFERENCE

**1) MKT_UNIQUE_CARRIER:** There are 10 carriers, with AA being the most frequent (~50k flights), indicating one airline dominates the dataset.

**2) ORIGIN & DEST:** Around 358–359 airports are present, showing good geographic diversity; ATL is the most common hub for both origin and destination.

**3)DEP_TIME_BLK:** There are 19 time blocks, with 0700–0759 being the busiest departure window, suggesting peak morning traffic.


## Checking for missing values

In [ ]:
df.isna().sum()

In [ ]:
sns.heatmap(df.isnull(),cmap='coolwarm',cbar=False)
plt.show()

### INFERENCE

**There are no null values present in the data**


## Checking For Outliers

In [ ]:
q3=df[num].quantile(0.75)
q1=df[num].quantile(0.25)
IQR =q3-q1

upper=q3 + 1.5*IQR
lower=q1 - 1.5*IQR

In [ ]:
df[((df[num]>upper)|(df[num]<lower)).any(axis=1)]

In [ ]:
# Checking outliers by plotting
plt.figure(figsize=(18,28))
t=1

for i in num:
    plt.subplot(7,3,t)
    sns.boxplot(df[i])
    t+=1
plt.tight_layout()
plt.show()

### INFERENCE 
**There are 62807 records in the data which are outliers.**

## Univariate Analysis

In [ ]:
# Numerical column Ploting
plt.figure(figsize=(18,28))
t=1
for i in num:
    plt.subplot(7,3,t)
    sns.histplot(df[i],kde=True)
    t+=1
plt.tight_layout()
plt.show()

### INFERENCE


In [ ]:
# Categorical column Ploting
plt.figure(figsize=(15,12))
t=1
for i in cat:
    plt.subplot(2,2,t)
    sns.histplot(df[i],kde=True)
    plt.xticks(rotation=90)
    t+=1
plt.tight_layout()
plt.show()

### INFERENCE

## Bivariate Analysis

In [ ]:
# Numrical vs Numerical Ploting
plt.figure(figsize=(18,28))
t=1
for i in num:
    plt.subplot(7,3,t)
    sns.scatterplot(df,x=i,y=df['OPERATIONAL_RISK'])
    t+=1
plt.tight_layout()
plt.show()

### INFERENCE

In [ ]:
# Categorical vs Numerical Ploting
plt.figure(figsize=(15,8))
t=1
for i in cat:
    plt.subplot(2,2,t)
    sns.barplot(df,x=i,y=df['OPERATIONAL_RISK'])
    plt.xticks(rotation=90)
    t+=1
plt.tight_layout()
plt.show()

### INFERENCE

## Multivariate Analysis

In [ ]:
# Pairplot
plt.figure(figsize=(15,8))
sns.pairplot(df[num])
plt.tight_layout()
plt.show()

### INFERENCE

In [ ]:
# Heatmap
dropped_col=df.drop(columns=['YEAR','ORIGIN_AIRPORT_ID','DEST_AIRPORT_ID','CANCELLED','DIVERTED'])
plt.figure(figsize=(15,8))

sns.heatmap(round(dropped_col.corr(numeric_only=True),3),annot=True,cmap='viridis')
plt.show()

### INFERENCE

1) NAS_DELAY and LATE_AIRCRAFT_DELAY have the highest positive correlation (~0.36) with operational risk, indicating delays from airspace and late aircraft are major risk drivers.

2) CARRIER_DELAY (~0.27) and TAXI_OUT (~0.26) show moderate positive relationships, suggesting airline-related delays and ground congestion also increase risk.

3) QUARTER, MONTH, DAY_OF_WEEK, AIR_TIME, DISTANCE variables have very weak correlation with operational risk, indicating they contribute less to predicting risk.



## Data Preprocessing


In [ ]:
# Creating copy of df
df_model=df.copy()

In [46]:
le=LabelEncoder()
df_model['MKT_UNIQUE_CARRIER']=le.fit_transform(df_model['MKT_UNIQUE_CARRIER'])


In [44]:
df.columns

Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_WEEK', 'MKT_UNIQUE_CARRIER',
       'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID', 'DEST',
       'CRS_DEP_TIME', 'DEP_TIME_BLK', 'TAXI_OUT', 'TAXI_IN', 'CANCELLED',
       'DIVERTED', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE',
       'DISTANCE_GROUP', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
       'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'OPERATIONAL_RISK'],
      dtype='str')

## Model Building

In [72]:
### Baseline Model
features=df_model.drop(
    columns=['OPERATIONAL_RISK','YEAR','ORIGIN','DEST','CANCELLED','DIVERTED','DEP_TIME_BLK','WEATHER_DELAY','NAS_DELAY',\
        'SECURITY_DELAY','LATE_AIRCRAFT_DELAY','ACTUAL_ELAPSED_TIME','AIR_TIME','QUARTER','CARRIER_DELAY'])

x=features
y=df_model['OPERATIONAL_RISK']

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

model=LogisticRegression()
model.fit(x_train,y_train)

ypred_soft=model.predict_proba(x_test)[:,1]
ypred_hard=(ypred_soft>0.5)

print(classification_report(y_test,ypred_hard))
print()
print('Confusion Matrix')
print(confusion_matrix(y_test,ypred_hard))
print()
print('Roc_auc score',roc_auc_score(y_test,ypred_soft))

              precision    recall  f1-score   support

           0       0.81      0.98      0.89     31579
           1       0.70      0.14      0.24      8422

    accuracy                           0.81     40001
   macro avg       0.76      0.56      0.56     40001
weighted avg       0.79      0.81      0.75     40001


Confusion Matrix
[[31073   506]
 [ 7215  1207]]

Roc_auc score 0.6773547122256418
